# Multiple Input and Multiple Output Channels

In [1]:
import torch
import torch.nn.functional as F
from d2l import torch as d2l

## Multiple Input Channels

In [2]:
def corr2d_multi_in(X, K):
# Iterate through the 0th dimension (channel) of K first, then add them up
    return sum(d2l.corr2d(x, k) for x, k in zip(X, K))

In [3]:
X = torch.tensor([[[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]],
               [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]])
K = torch.tensor([[[0.0, 1.0], [2.0, 3.0]], [[1.0, 2.0], [3.0, 4.0]]])
X, "", K

(tensor([[[0., 1., 2.],
          [3., 4., 5.],
          [6., 7., 8.]],
 
         [[1., 2., 3.],
          [4., 5., 6.],
          [7., 8., 9.]]]),
 '',
 tensor([[[0., 1.],
          [2., 3.]],
 
         [[1., 2.],
          [3., 4.]]]))

In [4]:
corr2d_multi_in(X, K)

tensor([[ 56.,  72.],
        [104., 120.]])

In [5]:
def test_xk(X, K):
    for x, k in zip(X, K):
        print(x, '\n', k, '\n')    
        
test_xk(X, K)

tensor([[0., 1., 2.],
        [3., 4., 5.],
        [6., 7., 8.]]) 
 tensor([[0., 1.],
        [2., 3.]]) 

tensor([[1., 2., 3.],
        [4., 5., 6.],
        [7., 8., 9.]]) 
 tensor([[1., 2.],
        [3., 4.]]) 



## Multiple Output Channels

In [6]:
def corr2d_multi_in_out(X, K):
    # Iterate through the 0th dimension of K, and each time, perform
    # cross-correlation operations with input X. All of the results are
    # stacked together
    return torch.stack([corr2d_multi_in(X, k) for k in K], 0)

In [7]:
torch.stack((K, K + 1, K + 2), 1), torch.stack((K, K + 1, K + 2), 1).size()

(tensor([[[[0., 1.],
           [2., 3.]],
 
          [[1., 2.],
           [3., 4.]],
 
          [[2., 3.],
           [4., 5.]]],
 
 
         [[[1., 2.],
           [3., 4.]],
 
          [[2., 3.],
           [4., 5.]],
 
          [[3., 4.],
           [5., 6.]]]]),
 torch.Size([2, 3, 2, 2]))

In [8]:
torch.stack((K, K + 1, K + 2), 2), torch.stack((K, K + 1, K + 2), 2).size()

(tensor([[[[0., 1.],
           [1., 2.],
           [2., 3.]],
 
          [[2., 3.],
           [3., 4.],
           [4., 5.]]],
 
 
         [[[1., 2.],
           [2., 3.],
           [3., 4.]],
 
          [[3., 4.],
           [4., 5.],
           [5., 6.]]]]),
 torch.Size([2, 2, 3, 2]))

In [9]:
K = torch.stack((K, K + 1, K + 2), 0)
K.shape, '', K

(torch.Size([3, 2, 2, 2]),
 '',
 tensor([[[[0., 1.],
           [2., 3.]],
 
          [[1., 2.],
           [3., 4.]]],
 
 
         [[[1., 2.],
           [3., 4.]],
 
          [[2., 3.],
           [4., 5.]]],
 
 
         [[[2., 3.],
           [4., 5.]],
 
          [[3., 4.],
           [5., 6.]]]]))

In [10]:
corr2d_multi_in_out(X, K)

tensor([[[ 56.,  72.],
         [104., 120.]],

        [[ 76., 100.],
         [148., 172.]],

        [[ 96., 128.],
         [192., 224.]]])

### 1 x 1 Covolutional Layer

In [11]:
def corr2d_multi_in_out_1x1(X, K):
    c_i, h, w = X.shape
    c_o = K.shape[0]
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K, X)
    return Y.reshape((c_o, h, w))

In [12]:
X = torch.normal(0, 1, (3, 3, 3))
K = torch.normal(0, 1, (2, 3, 1, 1))
Y1 = corr2d_multi_in_out_1x1(X, K)
Y2 = corr2d_multi_in_out(X, K)
assert float(torch.abs(Y1 - Y2).sum()) < 1e-6

In [13]:
X, '', K

(tensor([[[-1.3514, -0.4189, -0.1624],
          [ 0.2635, -0.3519,  1.0264],
          [ 0.5639,  1.7813, -0.1348]],
 
         [[-0.1595,  2.8485, -0.7813],
          [ 0.4634,  1.2053,  1.2012],
          [-0.8304, -0.2490,  1.3276]],
 
         [[-0.8849,  0.6966,  0.4607],
          [ 1.8131,  0.0639, -0.5734],
          [-1.4535, -2.2394, -2.1844]]]),
 '',
 tensor([[[[0.9501]],
 
          [[2.0857]],
 
          [[1.1801]]],
 
 
         [[[0.0715]],
 
          [[0.9969]],
 
          [[0.6029]]]]))

In [14]:
def test_1x1(X, K):
    c_i, h, w = X.shape
    print(c_i, h, w)
    c_o = K.shape[0]
    print(c_o)
    X = X.reshape((c_i, h * w))
    K = K.reshape((c_o, c_i))
    print(X, '\n', K)
    # Matrix multiplication in the fully connected layer
    Y = torch.matmul(K, X)
    print(Y)
    return Y.reshape((c_o, h, w))

In [15]:
Y_test = test_1x1(X, K)

3 3 3
2
tensor([[-1.3514, -0.4189, -0.1624,  0.2635, -0.3519,  1.0264,  0.5639,  1.7813,
         -0.1348],
        [-0.1595,  2.8485, -0.7813,  0.4634,  1.2053,  1.2012, -0.8304, -0.2490,
          1.3276],
        [-0.8849,  0.6966,  0.4607,  1.8131,  0.0639, -0.5734, -1.4535, -2.2394,
         -2.1844]]) 
 tensor([[0.9501, 2.0857, 1.1801],
        [0.0715, 0.9969, 0.6029]])
tensor([[-2.6608e+00,  6.3652e+00, -1.2403e+00,  3.3565e+00,  2.2550e+00,
          2.8040e+00, -2.9114e+00, -1.4694e+00,  6.3227e-02],
        [-7.8909e-01,  3.2297e+00, -5.1280e-01,  1.5739e+00,  1.2149e+00,
          9.2529e-01, -1.6638e+00, -1.4708e+00, -3.0407e-03]])


In [16]:
Y_test

tensor([[[-2.6608e+00,  6.3652e+00, -1.2403e+00],
         [ 3.3565e+00,  2.2550e+00,  2.8040e+00],
         [-2.9114e+00, -1.4694e+00,  6.3227e-02]],

        [[-7.8909e-01,  3.2297e+00, -5.1280e-01],
         [ 1.5739e+00,  1.2149e+00,  9.2529e-01],
         [-1.6638e+00, -1.4708e+00, -3.0407e-03]]])

## Exercises

### Ex. 1.1

In [17]:
def corr2d(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

In [18]:
X = torch.randn(size=(8, 8))
K_1 = torch.randn(2, 2)
K_1_1 = torch.randn(3, 3)

In [19]:
K_2 = F.conv_transpose2d(K_1.reshape(1,1,K_1.shape[0],-1), K_1_1.reshape(1,1,K_1_1.shape[0],-1)).squeeze()
K_2

tensor([[-8.0468e-02, -5.8805e-01,  1.5210e+00, -9.7891e-01],
        [ 3.5296e-01, -2.0957e+00,  3.2617e+00, -7.1371e-01],
        [-7.1843e-02, -3.3753e-04, -8.5613e-02,  3.3048e+00],
        [ 1.6262e-01, -1.0312e+00,  9.1893e-01,  2.3821e+00]])

In [20]:
K_1, '', K_1_1

(tensor([[-0.0828, -0.7825],
         [ 0.3552, -1.4430]]),
 '',
 tensor([[ 0.9714, -2.0763,  1.2510],
         [-0.0955,  0.3763, -1.3951],
         [ 0.4578, -1.0432, -1.6508]]))

In [21]:
print(K_1.shape,K_1_1.shape,K_2.shape)
print((corr2d(X,K_2)-corr2d(corr2d(X,K_1),K_1_1)< 1e-6).all())

torch.Size([2, 2]) torch.Size([3, 3]) torch.Size([4, 4])
tensor(False)


### Ex.4

In [23]:
(Y1==Y2).all()

tensor(True)